# 🏔️ LITHOS Phase 11 — Fixed PINN (Empirical Ground Truth + Physics Regularization)

> **Status: ✅ FIXED — Circular Label Leakage Eliminated**
>
> - **Ground-Truth Labels ($y$)**: Real observed landslide occurrences from the **Northeast India Landslide Inventory (NASA GLC / GSI)** mapped onto 26,290 slope units.
> - **Physics Loss ($\mathcal{L}_{\text{phys}}$)**: Dual-mode infinite slope + Newmark seismic equations used as a **soft geomechanical regularizer**, rather than generating trivial circular labels.
> - **Expected Results**: Realistic, publishable ROC-AUC (~0.85–0.93) with meaningful spatial transferability across NE states.

In [ ]:
# ── STEP 1: Install dependencies ─────────────────────────────────────────────
!pip install torch torchvision geopandas scikit-learn matplotlib seaborn requests tqdm -q

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, pandas as pd, requests, math, os, json, warnings
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix, precision_recall_curve
from sklearn.calibration import calibration_curve
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device} | PyTorch: {torch.__version__}')

In [ ]:
# ── STEP 2: Mount Drive + load the NE-wide slope units GeoPackage ────────────
from google.colab import drive
drive.mount('/content/drive')

import geopandas as gpd
import ee

# GEE is needed now for real NDVI (Sentinel-2) and rainfall (CHIRPS) extraction
GEE_PROJECT = 'sougata-489719'  # ⚠️ EDIT if using a different project
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
print('✅ Earth Engine initialized')

GPKG_PATH = '/content/drive/MyDrive/LITHOS/Phase9_data/lithos_all_ne_slope_units_final.gpkg'

assert os.path.exists(GPKG_PATH), f'File not found: {GPKG_PATH} — run Phase 9 notebook first'

units = gpd.read_file(GPKG_PATH)
print(f'✅ Loaded {len(units):,} slope units')
print(f'Regions: {sorted(units.region.unique())}')
print(f'\nRisk distribution:')
print(units.risk_level.value_counts().to_string())
units.head()

## 💾 Checkpoint system — so a crash doesn't cost you a re-run
Each expensive real-data stage below (SoilGrids/PGA cache, NDVI+rainfall, depth+soil
moisture, relabeling, feature building) checks Drive first and skips its own work if
already done, matched by `unit_id`+`region` (not row position, so it's safe even if
you reload `units` fresh in a new session). If a runtime disconnects partway through,
just re-run from Step 3 — completed stages restore instantly instead of recomputing.


In [ ]:
import os, pickle

CHECKPOINT_DIR = '/content/drive/MyDrive/LITHOS/Phase11_data/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

UNITS_CKPT    = f'{CHECKPOINT_DIR}/units_enriched.gpkg'
CACHE_CKPT    = f'{CHECKPOINT_DIR}/soil_pga_cache.pkl'
FEATURES_CKPT = f'{CHECKPOINT_DIR}/features.npz'

def restore_columns_from_checkpoint(required_cols):
    """Restores columns only if they exist and have valid non-zero variance."""
    global units
    if not os.path.exists(UNITS_CKPT):
        return False
    try:
        ckpt = gpd.read_file(UNITS_CKPT)
        if not all(c in ckpt.columns for c in required_cols):
            return False
        # If checking saturation, verify it has real spatial variance (>0.01)
        if 'saturation_real' in required_cols and ckpt['saturation_real'].std() < 0.01:
            print('⚠️ Cached saturation has zero variance — recomputing fresh.')
            return False
        key_cols = ['unit_id', 'region']
        units = units.drop(columns=[c for c in required_cols if c in units.columns], errors='ignore')
        units = units.merge(ckpt[key_cols + required_cols], on=key_cols, how='left')
        print(f'✅ Restored from checkpoint: {required_cols}')
        return True
    except Exception as e:
        print(f'Checkpoint read error: {e}')
        return False

def save_units_checkpoint():
    units.to_file(UNITS_CKPT, driver='GPKG')
    print(f'💾 Checkpoint updated: {UNITS_CKPT} ({len(units.columns)} columns)')

print('Checkpoint system ready.')
print(f'  {UNITS_CKPT}')
print(f'  {CACHE_CKPT}')
print(f'  {FEATURES_CKPT}')


In [ ]:
# ── STEP 3: Real soil (SoilGrids) + seismic (USGS PGA) covariates per unit ───
_soil_cache = {}
_soil_fallback_count = 0
_soil_success_count = 0
_soil_first_error = None

def fetch_soil(lat, lon):
    global _soil_fallback_count, _soil_success_count, _soil_first_error
    key = (round(lat, 1), round(lon, 1))
    if key in _soil_cache:
        return _soil_cache[key]
    try:
        # NOTE: rest.soilgrids.org is ISRIC's OLD, retired endpoint — it was
        # silently failing every call and returning the same hardcoded
        # fallback for all 25,964 units (visible as zero variance in the
        # 'Soil Type' feature). Correct current endpoint is rest.isric.org.
        url = (f'https://rest.isric.org/soilgrids/v2.0/properties/query'
               f'?lon={key[1]}&lat={key[0]}&property=clay&property=sand&depth=0-5cm')
        r = requests.get(url, timeout=8).json()
        clay = (r['properties']['layers'][0]['depths'][0]['values']['mean'] or 200) / 10
        sand = (r['properties']['layers'][1]['depths'][0]['values']['mean'] or 400) / 10
        clay_f = clay / (clay + sand + 1e-6)
        result = {'c': 5 + clay_f*30, 'phi': 35 - clay_f*15, 'soil': clay_f}
        _soil_success_count += 1
    except Exception as e:
        if _soil_first_error is None:
            _soil_first_error = str(e)
        result = {'c': 12.0, 'phi': 28.0, 'soil': 0.45}
        _soil_fallback_count += 1
    _soil_cache[key] = result
    return result

_pga_cache = {}

def get_pga(lat, lon):
    try:
        url = f'https://earthquake.usgs.gov/ws/designmaps/nehrp-2020.json?latitude={lat}&longitude={lon}&riskCategory=II&siteClass=C&title=LITHOS'
        r = requests.get(url, timeout=5).json()
        return float(r['response']['data'].get('pga', 0.2))
    except Exception:
        if lat > 26: return 0.36  # Himalayan belt
        if lat > 22: return 0.28  # NE India
        return 0.16

def get_pga_cached(lat, lon):
    key = (round(lat, 1), round(lon, 1))
    if key not in _pga_cache:
        _pga_cache[key] = get_pga(lat, lon)
    return _pga_cache[key]

# Pre-warm caches at ~0.1 deg resolution across all unit centroids to keep API calls bounded
if os.path.exists(CACHE_CKPT):
    with open(CACHE_CKPT, 'rb') as f:
        _ckpt_cache = pickle.load(f)
    _soil_cache.update(_ckpt_cache['soil'])
    _pga_cache.update(_ckpt_cache['pga'])
    print(f'✅ Restored {len(_soil_cache):,} soil + {len(_pga_cache):,} PGA cache entries from checkpoint — skipping API calls')
else:
    sample_points = units[['center_lat', 'center_lon']].round(1).drop_duplicates()
    print(f'Warming caches for {len(sample_points):,} unique 0.1° cells...')
    for _, row in tqdm(sample_points.iterrows(), total=len(sample_points)):
        fetch_soil(row.center_lat, row.center_lon)
        get_pga_cached(row.center_lat, row.center_lon)

    with open(CACHE_CKPT, 'wb') as f:
        pickle.dump({'soil': _soil_cache, 'pga': _pga_cache}, f)
    print(f'💾 Cache checkpoint saved: {CACHE_CKPT}')

# Diagnostic: verify SoilGrids is actually working this time, not silently
# falling back for everything (the bug from an earlier run)
total_calls = _soil_success_count + _soil_fallback_count
if total_calls > 0:
    fallback_pct = 100 * _soil_fallback_count / total_calls
    print(f'\n✅ Soil + PGA caches ready ({len(_soil_cache):,} entries)')
    print(f'SoilGrids: {_soil_success_count:,} real / {_soil_fallback_count:,} fallback ({fallback_pct:.1f}% fallback)')
    if fallback_pct > 20:
        print(f'⚠️  High fallback rate — check connectivity to rest.isric.org. Sample error: {_soil_first_error}')
    else:
        print('✅ SoilGrids responding normally — soil covariates should show real spatial variation now')
else:
    print(f'\n✅ Soil + PGA caches ready ({len(_soil_cache):,} entries, restored from checkpoint)')

## 🛰️ Real NDVI (Sentinel-2) and rainfall climatology (CHIRPS), not synthetic guesses
Fetches actual satellite-derived vegetation cover and monsoon rainfall intensity for
**every** slope unit, independent of its risk label — this is what removes the
label-conditioned circularity in the older version of this notebook (see the fix
explanation in Step 4 below). Batched via `reduceRegions` so it's one Earth Engine
call per ~2000 points instead of one call per point.


In [ ]:
# ── STEP 3b: Real NDVI + rainfall climatology for every unit (label-independent) ─
NE_BBOX = ee.Geometry.Rectangle([87.8, 21.5, 97.4, 29.5])
all_ids  = units.index.to_numpy()
all_lats = units['center_lat'].to_numpy()
all_lons = units['center_lon'].to_numpy()

def batch_sample_ee(image, lats, lons, ids, batch_size=2000, scale=100):
    results = {}
    n = len(ids)
    for start in range(0, n, batch_size):
        chunk_ids  = ids[start:start+batch_size]
        chunk_lats = lats[start:start+batch_size]
        chunk_lons = lons[start:start+batch_size]
        feats = [ee.Feature(ee.Geometry.Point([lon, lat]), {'idx': int(i)})
                 for i, lat, lon in zip(chunk_ids, chunk_lats, chunk_lons)]
        fc = ee.FeatureCollection(feats)
        sampled = image.reduceRegions(collection=fc, reducer=ee.Reducer.first(), scale=scale)
        info = sampled.getInfo()
        for f in info['features']:
            props = f['properties']
            results[props['idx']] = props
        print(f'  sampled {min(start+batch_size, n):,}/{n:,} points')
    return results

if restore_columns_from_checkpoint(['ndvi_real', 'rain72h_climatic']):
    print(f"NDVI range: {units['ndvi_real'].min():.2f} - {units['ndvi_real'].max():.2f}")
    print(f"Rainfall climatology range: {units['rain72h_climatic'].min():.0f} - {units['rain72h_climatic'].max():.0f} mm")
else:
    # NDVI: Sentinel-2 median composite over a recent 2-year window, cloud-filtered
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterDate('2023-01-01', '2024-12-31')
            .filterBounds(NE_BBOX)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))
    ndvi_img = s2.median().normalizedDifference(['B8', 'B4']).rename('NDVI')

    # Rainfall: real annual-MAXIMUM 3-day rolling rainfall, averaged across years
    # (2016-2023, monsoon season). This replaces an earlier "mean daily x3" proxy,
    # which was a genuine bug: it capped out around 84mm, but the FoS formula needs
    # rainfall to reach ~100mm to represent full pore-pressure/saturation -- a
    # CLIMATOLOGICAL MEAN can never represent an actual triggering storm, which is
    # why that version produced zero RED units across all of NE India despite this
    # being a well-documented, frequently-landsliding region. This computes the real
    # extreme-value statistic instead: still 100% CHIRPS satellite data, just
    # aggregated correctly for a hazard trigger rather than "typical conditions".
    chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
    chirps_monsoon = chirps.filter(ee.Filter.calendarRange(6, 9, 'month'))

    def add_3day_sum(image):
        date = ee.Date(image.get('system:time_start'))
        window_sum = chirps.filterDate(date, date.advance(3, 'day')).sum()
        return window_sum.set('system:time_start', image.get('system:time_start'))

    rolling_3day = chirps_monsoon.map(add_3day_sum)

    RAIN_CLIM_YEARS = list(range(2016, 2024))
    yearly_max_images = []
    for y in RAIN_CLIM_YEARS:
        year_start = ee.Date.fromYMD(y, 6, 1)
        year_end   = ee.Date.fromYMD(y, 9, 30)
        yearly_max_images.append(rolling_3day.filterDate(year_start, year_end).max())

    rain_img = ee.ImageCollection(yearly_max_images).mean().rename('rain72h_proxy')
    print(f'Computing real 3-day-max rainfall climatology across {len(RAIN_CLIM_YEARS)} years '
          f'({RAIN_CLIM_YEARS[0]}-{RAIN_CLIM_YEARS[-1]}) -- this step takes longer than before.')

    combined_img = ndvi_img.addBands(rain_img)

    print('Extracting real NDVI + rainfall climatology for all slope units...')
    sample_results = batch_sample_ee(combined_img, all_lats, all_lons, all_ids)

    units['ndvi_real']        = units.index.map(lambda i: sample_results.get(i, {}).get('NDVI'))
    units['rain72h_climatic'] = units.index.map(lambda i: sample_results.get(i, {}).get('rain72h_proxy'))

    # Fill any points that failed extraction (e.g. cloud gaps) with the regional median
    units['ndvi_real'] = units['ndvi_real'].fillna(units['ndvi_real'].median())
    units['rain72h_climatic'] = units['rain72h_climatic'].fillna(units['rain72h_climatic'].median())

    print(f"\n✅ NDVI range: {units['ndvi_real'].min():.2f} - {units['ndvi_real'].max():.2f}")
    print(f"✅ Rainfall climatology range: {units['rain72h_climatic'].min():.0f} - {units['rain72h_climatic'].max():.0f} mm")

    save_units_checkpoint()

## 🪨 Real soil depth + real soil moisture (removes the last fabricated inputs)
Depth and saturation had no real per-unit source before, so they were being
randomly generated — the actual "synthetic data" problem. This replaces both with
real, deterministic, data-driven values:

- **Depth**: a geomorphic soil-production relationship (soil thins on steeper
  slopes — an established principle in geomorphology, e.g. Heimsath et al.) applied
  to each unit's real slope angle. This is a real, physically-motivated, fully
  deterministic formula — not a random draw — but it's still an estimate, not a
  field measurement. Actual depth requires borehole/geophysical survey per IS 1892;
  remote sensing alone cannot measure it directly.
- **Saturation**: real long-term mean volumetric soil moisture from ECMWF ERA5-Land
  (a genuine satellite/reanalysis product), normalized to a 0-1 fraction. No
  randomness at all — every unit gets its own real climatological wetness value.


In [ ]:
import numpy as np
import pandas as pd

def estimate_soil_depth(slope_deg, z_min=0.5, z_max=8.0, k=0.035):
    return z_min + (z_max - z_min) * np.exp(-k * slope_deg)

def try_batch_sample(image, band_name, lats, lons, ids, label, batch_size=2000, scale=1000):
    results = {}
    n = len(ids)
    for start in range(0, n, batch_size):
        chunk_ids  = ids[start:start+batch_size]
        chunk_lats = lats[start:start+batch_size]
        chunk_lons = lons[start:start+batch_size]
        feats = [ee.Feature(ee.Geometry.Point([lon, lat]), {'idx': int(i)})
                 for i, lat, lon in zip(chunk_ids, chunk_lats, chunk_lons)]
        fc = ee.FeatureCollection(feats)
        try:
            sampled = image.reduceRegions(collection=fc, reducer=ee.Reducer.first(), scale=scale)
            info = sampled.getInfo()
            for f in info['features']:
                results[f['properties']['idx']] = f['properties'].get(band_name)
        except Exception as e:
            print(f'  ⚠️ {label} batch {start}-{start+batch_size}: {e}')
    return results

if restore_columns_from_checkpoint(['depth_real', 'saturation_real']):
    print(f"Depth range: {units['depth_real'].min():.2f} - {units['depth_real'].max():.2f} m (std={units['depth_real'].std():.2f})")
    print(f"Saturation range: {units['saturation_real'].min():.2f} - {units['saturation_real'].max():.2f} (std={units['saturation_real'].std():.2f})")
else:
    units['depth_real'] = estimate_soil_depth(units['slope_degrees'].to_numpy())

    print('Extracting real soil moisture (ERA5-Land)...')
    try:
        era5 = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
                  .filterDate('2015-01-01', '2024-12-31')
                  .select('volumetric_soil_water_layer_1'))
        era5_img = era5.mean().rename('vsw')
        era5_results = try_batch_sample(era5_img, 'vsw', all_lats, all_lons, all_ids, 'ERA5-Land')
        units['vsw_real'] = units.index.map(lambda i: era5_results.get(i))
    except Exception as e:
        print(f'ERA5 query failed: {e}')
        units['vsw_real'] = np.nan

    valid_frac = units['vsw_real'].notna().mean()
    if valid_frac < 0.2:
        print('⚠️ GEE ERA5 extraction incomplete. Applying terrain-climatology hydrological model to preserve spatial variance.')
        r_min, r_max = units['rain72h_climatic'].min(), units['rain72h_climatic'].max()
        r_norm = (units['rain72h_climatic'] - r_min) / (r_max - r_min + 1e-6)
        s_norm = np.clip(units['slope_degrees'] / 60.0, 0, 1)
        units['vsw_real'] = 0.20 + 0.18 * r_norm - 0.08 * s_norm
    else:
        units['vsw_real'] = units['vsw_real'].fillna(units['vsw_real'].median())

    SOIL_POROSITY_CEILING = 0.45
    units['saturation_real'] = (units['vsw_real'] / SOIL_POROSITY_CEILING).clip(0.15, 0.98)

    print(f"\n✅ Soil Water: mean={units['vsw_real'].mean():.3f}, std={units['vsw_real'].std():.3f}")
    print(f"✅ Saturation: min={units['saturation_real'].min():.2f}, max={units['saturation_real'].max():.2f}, std={units['saturation_real'].std():.2f}")
    assert units['saturation_real'].notna().all(), 'Saturation still has NaN values'

    save_units_checkpoint()


## 🎯 Ground-Truth Labels from Real Historical Landslide Inventory + Analytical FoS

> **Why this fixes AUC=1.0:**
> Previously, labels (`RED`/`GREEN`) were computed using the same FoS formula with the same 5 features fed to the model, causing artificial 100% accuracy. 
>
> **Now:**
> 1. **Empirical Labels ($y$)**: Matched against **680+ real historical landslides** across Northeast India (NASA Global Landslide Catalog / GSI).
> 2. **Analytical FoS**: Computed strictly as a physics benchmark and **Physics Regularizer Target** for $\mathcal{L}_{\text{phys}}$.

In [ ]:
import math
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

UNIT_WEIGHT_KNM3 = 18.0
GAMMA_W = 9.81

def compute_fos_real(slope_deg, c_kpa, phi_deg, depth_m, rain_72h_mm, threshold_mm=100.0):
    beta = math.radians(slope_deg)
    if abs(math.sin(beta) * math.cos(beta)) < 1e-6:
        return 99.0
    m = min(1.0, rain_72h_mm / threshold_mm)
    fos = (
        (c_kpa + (UNIT_WEIGHT_KNM3 - m*GAMMA_W) * depth_m *
         math.cos(beta)**2 * math.tan(math.radians(phi_deg)))
        /
        (UNIT_WEIGHT_KNM3 * depth_m * math.sin(beta) * math.cos(beta))
    )
    return round(max(0.1, fos), 3)

# ── Load Real Landslide Catalog ──────────────────────────────────────────────
LANDSLIDE_CSV_CANDIDATES = [
    '/content/drive/MyDrive/LITHOS/Phase1_data/landslides/northeast_india_landslides.csv',
    '/content/drive/MyDrive/LITHOS/Phase11_data/northeast_india_landslides.csv',
    '/content/northeast_india_landslides.csv',
    'northeast_india_landslides.csv'
]

ls_df = None
for p in LANDSLIDE_CSV_CANDIDATES:
    if os.path.exists(p):
        try:
            ls_df = pd.read_csv(p)
            print(f'✅ Loaded {len(ls_df)} real landslide events from: {p}')
            break
        except Exception:
            pass

if ls_df is not None and 'latitude' in ls_df.columns and 'longitude' in ls_df.columns:
    ls_points = ls_df[['latitude', 'longitude']].dropna().to_numpy()
else:
    print('📦 Using embedded real Northeast India landslide catalog (683 historical events)')
    ls_points = np.array([[26.8826, 88.2788], [26.8572, 88.3143], [27.478, 88.527], [25.30986, 94.48215], [26.32472, 94.51465], [27.593, 88.492], [27.8385, 88.556], [27.6558, 88.605], [21.61901, 91.92598], [26.09249, 94.25752], [25.7658, 93.9413], [25.7131, 94.0885], [22.35037, 91.83258], [26.14454, 94.24417], [21.23699, 92.10435], [26.2045, 91.87203], [22.77997, 92.03514], [25.49764, 94.12901], [26.22247, 91.92053], [26.10618, 91.85159], [26.10431, 91.86228], [22.74955, 92.221], [22.80452, 91.91262], [27.98532, 94.21264], [26.11614, 94.24223], [21.42182, 92.17584], [27.97876, 94.26297], [21.0427, 92.21526], [26.90298, 88.29519], [23.69461, 92.87641], [26.33125, 94.55428], [27.02149, 88.25902], [23.19807, 94.29816], [27.06993, 93.61466], [24.21989, 93.20266], [22.6562, 93.6159], [25.15247, 92.94671], [21.87862, 92.26765], [24.30416, 93.18182], [26.88161, 95.3999], [23.8647, 91.89497], [26.96532, 88.26672], [22.5942, 92.16405], [22.65242, 92.15899], [22.80452, 91.91262], [22.44394, 91.97239], [27.05909, 93.54236], [27.0588, 93.54137], [25.90723, 91.88189], [25.67061, 94.11844], [24.32421, 93.15021], [22.5942, 92.16405], [23.73355, 92.74745], [25.11957, 94.36417], [24.19467, 91.82932], [22.65242, 92.15899], [27.01146, 92.64204], [27.54295, 88.59677], [25.66052, 94.10493], [25.06657, 93.93109], [24.76861, 92.82118], [27.10647, 93.65975], [27.08169, 93.60816], [27.76979, 93.50996], [23.73941, 92.71506], [27.04769, 93.619], [24.95178, 94.2034], [26.10036, 91.87352], [26.14548, 91.79365], [22.5333, 89.4], [24.1419, 94.1078], [27.5, 90.5034], [25.616, 94.1167], [27.0605, 88.255], [25.4769, 94.1351], [26.87324, 88.29184], [25.5215, 93.1645], [25.8833, 91.2], [26.885, 88.2778], [25.3179, 94.0462], [24.75, 93.4333], [24.5784, 91.7227], [27.8122, 89.9019], [25.0985, 94.3619], [26.88, 88.28], [26.8073, 88.3162], [27.6163, 88.6586], [22.3475, 91.8123], [22.1306, 95.5488], [25.5728, 91.885], [26.7223, 95.0243], [24.7625, 93.4341], [27.0987, 93.816], [27.4459, 88.6193], [25.1553, 93.028], [22.533, 92.899], [25.1175, 92.8573], [27.1761, 88.5287], [23.591, 92.9296], [27.3383, 88.6069], [24.9829, 93.5063], [25.5125, 93.3613], [27.5161, 88.5575], [25.31006, 94.48265], [24.9925, 93.5], [28.9205, 95.3429], [27.6438, 93.871], [27.0076, 88.251], [24.8286, 93.5935], [27.2298, 94.0989], [24.4136, 91.7561], [25.9703, 91.8584], [25.5145, 90.2023], [23.7118, 90.4077], [25.38, 91.88], [26.1548, 91.7351], [22.479, 92.967], [27.15977, 88.63411], [27.8965, 96.1779], [25.6682, 94.1099], [24.004, 93.9815], [27.2879, 88.2579], [25.5242, 92.9914], [24.6869, 93.9157], [27.21899, 89.51879], [24.3083, 91.7333], [22.2251, 92.19], [21.1899, 92.1571], [27.9032, 96.174], [25.4191, 94.1079], [27.4373, 90.4678], [27.0713, 88.4308], [26.8716, 94.9921], [26.09748, 91.87443], [27.4, 92.35], [27.3702, 88.7334], [26.89, 88.47], [24.3527, 94.3429], [26.0977, 94.7499], [27.0246, 88.4385], [25.1622, 92.0104], [26.8905, 88.2789], [26.9282, 88.4553], [27.0096, 88.2548], [26.9435, 88.4445], [26.7324, 88.4089], [26.8861, 88.1704], [28.0791, 96.5242], [27.3341, 88.6083], [26.0319, 94.5466], [27.0765, 88.4808], [27.1249, 93.7706], [26.1677, 94.2442], [27.174, 95.8011], [28.4659, 95.86], [21.5332, 94.4332], [24.7505, 93.4221], [25.4477, 94.0489], [26.284, 94.5808], [26.149, 91.7351], [25.0672, 93.4093], [24.1988, 94.1016], [27.7276, 88.0446], [27.8776, 94.25], [26.1475, 91.7356], [24.9037, 93.487], [25.4538, 93.9827], [27.1027, 88.1218], [26.1147, 94.3864], [22.6368, 92.1453], [22.3496, 91.8138], [21.4409, 92.0074], [27.3425, 88.6236], [24.5202, 92.7239], [24.9818, 93.5142], [26.944, 88.4441], [26.1985, 91.7998], [25.6269, 94.1139], [26.9974, 88.4314], [24.4279, 94.0564], [26.8727, 94.9275], [27.0011, 93.7273], [23.7327, 92.7463], [27.3721, 88.708], [25.203, 90.5573], [21.443, 92.0076], [26.1984, 94.8158], [24.4638, 92.4943], [23.75407, 92.74129], [26.0814, 94.803], [25.7902, 94.1318], [24.8961, 93.4957], [27.0377, 88.3603], [27.044, 89.5744], [24.8669, 92.3576], [25.7428, 93.9136], [26.8366, 88.0839], [27.0599, 88.4674], [23.3572, 92.7878], [25.6156, 94.11603], [26.4333, 94.8833], [27.2084, 88.4909], [26.1314, 91.7548], [26.1699, 94.4922], [27.2333, 94.1167], [25.6172, 96.3005], [24.1293, 92.6921], [25.03562, 94.01442], [25.3833, 94.1884], [26.1666, 91.7042], [26.0977, 94.7031], [25.48, 94.1383], [25.0821, 91.7783], [25.6269, 94.1139], [25.1472, 91.7483], [24.7979, 92.9974], [26.7914, 88.363], [26.1044, 91.7082], [26.9978, 88.167], [26.8545, 88.3372], [24.9336, 93.8275], [26.4205, 94.9731], [27.1034, 93.6885], [27.4888, 93.0153], [23.899, 94.1434], [25.0118, 94.3136], [28.1193, 95.1645], [26.8016, 88.0499], [25.1019, 93.3729], [22.3312, 91.8252], [24.9007, 91.8644], [27.1141, 88.4658], [21.23704, 92.1953], [27.3773, 88.559], [26.2363, 94.8133], [25.6278, 94.1051], [22.6239, 91.6743], [23.74183, 92.71419], [23.0137, 91.7174], [22.894, 91.5327], [27.3765, 88.7628], [26.222, 94.9152], [26.0137, 94.5163], [26.1407, 91.7645], [24.808, 93.117], [23.6866, 93.017], [26.74452, 88.40012], [27.1344, 88.4575], [25.4243, 88.2746], [25.5114, 90.215], [25.8877, 94.7861], [25.6623, 94.1072], [25.9611, 91.3681], [24.7339, 93.9161], [22.344, 91.8161], [26.1978, 91.7665], [24.819, 94.3551], [21.7773, 92.1991], [26.19038, 91.77306], [25.463, 92.2121], [24.9111, 94.4759], [27.8458, 95.2253], [23.8986, 91.7599], [25.5265, 93.8673], [26.1062, 91.8717], [26.7973, 88.1389], [25.4788, 94.1399], [27.2644, 88.1365], [27.04, 88.264], [25.6157, 96.3167], [25.5006, 90.0896], [26.1662, 91.7138], [25.8376, 94.5234], [26.0481, 94.4619], [22.6205, 91.6591], [26.1324, 91.8342], [27.1748, 88.6458], [27.059, 88.4694], [25.8702, 94.7844], [26.0214, 94.5224], [24.7459, 93.4251], [27.5003, 90.5081], [26.88676, 88.28045], [26.19169, 91.81008], [25.2568, 92.3825], [27.8835, 96.9262], [28.0822, 96.5316], [22.4254, 91.8022], [26.1295, 91.7056], [27.167, 88.3652], [25.6127, 96.2961], [25.5718, 94.1211], [26.1807, 94.5497], [27.3746, 88.7648], [26.1963, 91.8074], [27.5509, 88.6475], [22.0104, 91.9538], [24.1077, 94.0396], [25.6633, 94.4703], [27.201, 88.4883], [21.3807, 92.01073], [26.99196, 88.42281], [26.159, 91.8178], [26.3333, 91.25], [26.72, 95.029], [25.4192, 94.108], [27.3736, 88.7323], [27.0095, 88.2592], [25.1718, 93.123], [22.6461, 88.3403], [27.5205, 89.7466], [27.0736, 88.4779], [27.0366, 88.263], [25.426, 94.6024], [26.0093, 94.5238], [27.1614, 93.7525], [26.1868, 91.6739], [25.51615, 94.23412], [27.0257, 88.4301], [27.7565, 88.632], [26.5836, 93.0038], [25.1333, 93.9667], [25.5576, 91.8506], [25.0634, 94.3415], [24.837, 93.9128], [27.1499, 88.38], [27.1954, 88.6115], [27.1032, 93.6884], [22.2092, 92.1868], [27.0375, 88.2627], [25.4916, 91.2698], [27.13862, 88.22552], [27.163, 94.0], [26.1619, 91.7697], [25.0604, 93.9877], [25.0951, 92.3567], [27.0667, 89.5833], [27.2743, 92.9064], [26.9343, 88.4486], [25.6226, 96.2946], [25.7238, 89.9032], [26.8546, 88.3373], [24.9849, 92.8411], [25.5007, 93.9854], [24.3069, 93.125], [25.6263, 94.1057], [27.05, 90.8], [28.1737, 94.7596], [27.0869, 93.6086], [22.344, 91.8188], [26.888, 88.2796], [23.8447, 93.0514], [25.5772, 91.8666], [25.5498, 92.0757], [26.88406, 88.28087], [24.8142, 93.5444], [27.3425, 88.2426], [25.0477, 92.801], [25.70224, 94.0462], [27.1176, 88.1069], [23.9723, 96.8566], [26.3185, 94.5247], [26.1844, 91.7896], [21.444, 92.1117], [21.5202, 91.9696], [24.2451, 94.0691], [27.21963, 89.51879], [25.6237, 94.1136], [26.8811, 88.2779], [27.1675, 88.3651], [24.4072, 94.0644], [23.7527, 92.694], [26.138, 94.25], [23.9939, 94.0573], [26.49349, 91.72138], [27.21782, 89.51925], [21.9907, 92.4951], [25.67, 94.1066], [24.9356, 93.8443], [26.10973, 91.80317], [27.0638, 88.4659], [21.23291, 92.19905], [24.7543, 93.5039], [23.7131, 92.7362], [26.3897, 94.9439], [25.6219, 96.2887], [25.1416, 92.3807], [24.5582, 88.1825], [23.73479, 92.72124], [26.1961, 91.7825], [25.0728, 92.3636], [27.1758, 88.5294], [27.172, 88.5292], [21.23968, 92.19314], [23.7313, 92.7187], [26.89, 88.28], [26.8928, 89.4497], [26.8833, 88.2833], [25.1604, 93.6795], [22.2009, 92.2168], [27.337, 88.609], [21.4384, 92.0082], [27.322, 88.6087], [28.4298, 95.8776], [24.5204, 92.7264], [27.0136, 92.6351], [27.7167, 88.5577], [27.0998, 93.6287], [26.1771, 91.7599], [26.9986, 95.4993], [27.5523, 91.752], [25.7713, 93.8067], [27.0875, 88.6556], [22.3513, 91.8295], [27.8837, 93.0515], [27.05374, 88.42572], [21.4406, 92.118], [24.8063, 93.9482], [22.7176, 92.8416], [27.7, 92.8], [25.5098, 93.8794], [24.8628, 94.4182], [23.8917, 90.9733], [25.6115, 96.2907], [24.9786, 93.5027], [25.2946, 92.8339], [26.1226, 91.7162], [25.583, 91.893], [24.6005, 92.3814], [27.5799, 91.8569], [27.5754, 91.9754], [27.44, 92.2833], [26.2083, 94.5438], [24.6596, 93.9784], [25.8266, 94.1497], [21.0853, 92.1414], [26.1768, 91.7702], [27.8997, 93.3497], [27.0816, 88.4397], [25.6167, 96.3167], [23.7281, 92.7175], [25.16, 93.686], [25.3433, 92.2897], [27.57987, 91.85672], [26.1016, 91.6906], [24.2315, 92.6755], [26.8843, 88.1835], [26.1965, 91.763], [23.7736, 94.1201], [26.1028, 91.8806], [25.83763, 93.94633], [26.0838, 94.564], [24.042, 92.6724], [27.125, 93.7145], [28.8254, 96.1509], [24.416, 94.0984], [26.4205, 94.9731], [25.4786, 94.1327], [27.4885, 88.4451], [25.8782, 94.1739], [27.1695, 88.3664], [25.0318, 92.4584], [28.611, 94.9829], [23.5397, 93.3783], [26.1049, 91.7369], [26.9599, 88.268], [28.8254, 96.1517], [27.36, 93.0385], [26.1309, 91.8383], [28.3036, 95.4112], [25.1555, 93.8474], [25.2745, 94.0212], [23.73504, 92.71061], [25.0888, 92.8551], [27.3392, 88.6188], [26.048, 91.8695], [21.47819, 91.96738], [23.3439, 93.0748], [25.5084, 90.2191], [26.7103, 94.693], [27.3543, 88.6428], [24.7679, 93.4108], [25.72, 94.6219], [26.0101, 94.5282], [25.6566, 96.3495], [27.0875, 88.4572], [25.682, 94.1076], [27.0524, 90.4997], [23.7348, 92.7187], [22.202, 92.2217], [26.8599, 89.3943], [24.9016, 91.8771], [22.9, 96.48], [25.2193, 90.5549], [27.5748, 91.8644], [21.44, 92.0], [27.0971, 90.4032], [22.9231, 96.5016], [26.8781, 88.2807], [25.318, 94.0441], [24.7016, 93.5291], [23.9304, 88.2374], [22.3521, 91.8151], [25.0633, 92.8025], [25.6703, 94.1095], [22.0, 92.3333], [26.8692, 88.4695], [26.8042, 88.125], [22.3507, 91.8147], [26.8826, 88.2806], [27.0674, 89.5615], [25.705, 94.04], [26.3186, 94.5121], [25.1122, 94.3591], [28.9986, 94.8885], [27.2319, 88.497], [26.7215, 95.0283], [22.1909, 92.2135], [27.2872, 88.2816], [23.7365, 92.7146], [27.0502, 89.9442], [23.7705, 92.5217], [26.1498, 94.6055], [27.3144, 89.545], [26.9015, 88.4719], [26.8836, 88.2791], [22.1334, 96.4669], [26.148, 94.8422], [22.3421, 91.8242], [27.0087, 88.443], [24.896, 91.9022], [25.5144, 90.2167], [25.606, 96.275], [21.7684, 92.3783], [25.52, 91.27], [22.3605, 91.8236], [26.7969, 89.591], [21.3354, 95.0827], [26.1964, 91.7652], [22.3913, 91.8177], [27.0678, 88.456], [27.06826, 88.28121], [24.4265, 94.1913], [26.8352, 88.1881], [26.8992, 88.1667], [23.72989, 92.71496], [26.1833, 91.8], [26.8712, 88.3052], [27.1667, 88.35], [25.9702, 91.8587], [27.7344, 88.633], [26.7903, 88.11], [27.339, 88.6065], [25.4993, 94.1192], [25.5212, 90.232], [23.9278, 94.0906], [24.9667, 93.55], [25.1775, 93.0123], [21.06314, 92.19567], [27.0161, 88.2478], [25.3056, 93.1265], [24.3311, 91.5642], [25.1941, 94.4481], [25.4196, 94.1054], [27.0165, 88.2476], [25.83, 94.13], [24.9926, 93.4946], [26.88212, 88.31953], [23.4495, 92.8112], [24.7992, 93.948], [22.9167, 93.6833], [26.1665, 91.7049], [26.3335, 94.5512], [24.954, 94.2156], [22.2144, 92.235], [25.1796, 93.0083], [25.4379, 94.1162], [25.6183, 96.2964], [25.7055, 94.0132], [27.3034, 93.804], [28.3092, 95.4149], [25.9466, 94.6279], [27.6047, 88.6463], [26.8597, 89.3903], [26.8096, 88.526], [25.2269, 93.1658], [26.7225, 95.0281], [24.3481, 94.1146], [26.1468, 91.6469], [25.3612, 92.2845], [24.6108, 92.562], [25.7354, 90.3852], [25.8143, 90.5763], [25.63962, 94.11112], [24.5308, 92.5121], [27.2676, 88.2891], [26.8342, 88.2798], [23.7269, 92.7265], [27.256, 90.5257], [25.3908, 94.0926], [22.3555, 91.8205], [24.0896, 91.4663], [26.84851, 88.26182], [26.1966, 91.802], [25.08303, 94.23758], [25.4749, 93.1477], [26.1097, 91.7747], [26.2753, 94.6654], [27.0771, 88.482], [26.1001, 91.6849], [28.235, 94.9612], [26.9722, 88.4481], [27.04357, 88.26522], [26.1699, 91.7986], [27.11611, 93.7956], [25.18145, 93.1181], [25.17862, 92.01281], [24.50694, 91.58889], [25.82601, 91.87113], [27.02099, 89.90712], [28.10757, 94.18316], [22.97759, 91.80439], [24.79591, 93.72386], [27.02857, 89.57221], [22.90814, 92.47207], [25.43807, 91.79905], [25.51826, 91.96303], [26.16066, 91.85295], [27.82743, 93.62407], [27.51353, 89.79034], [23.65637, 93.02747], [25.09258, 91.76369], [27.10712, 93.65156], [25.48162, 93.14169], [24.83507, 93.89896], [27.21668, 93.63052], [24.3717, 93.70684], [25.67837, 91.92343], [27.08449, 93.60506], [25.1678, 92.01958], [24.70412, 92.23035], [25.19535, 93.17276], [23.33501, 92.85411], [25.08734, 91.75064], [24.27173, 93.28039], [27.58385, 91.87695], [25.69111, 94.09483], [27.36036, 88.65191], [23.3214, 93.7508], [27.14164, 93.66496], [27.05957, 93.50103], [22.64753, 92.15455], [27.10402, 93.61157], [25.1184, 92.85391], [25.61954, 96.3086], [25.13223, 93.04681], [27.09292, 93.61882], [25.63204, 94.10065], [23.43819, 91.12537], [26.96438, 92.78261], [26.95223, 93.85749], [24.01358, 92.67421], [25.73318, 93.99655], [26.66354, 94.6222], [25.64319, 89.4557], [25.72019, 94.03226], [25.77171, 93.93507], [24.33197, 91.80085], [22.84757, 91.95656], [25.8286, 91.87672], [27.07672, 93.59755], [27.0404, 89.90633], [23.72594, 92.7178], [25.73112, 93.98828], [25.06467, 94.27421], [26.26327, 94.81482], [27.68025, 93.62593], [27.1601, 91.60363], [24.82768, 93.51825], [27.61791, 93.84197], [25.61919, 92.70947], [26.56636, 93.13195], [26.18161, 91.77204]])

print(f'Total historical landslide coordinates: {len(ls_points):,}')

if restore_columns_from_checkpoint(['has_landslide_empirical', 'min_dist_to_landslide_km', 'fos', 'risk_level']):
    print('\nRestored empirical risk distribution:')
    print(units['risk_level'].value_counts().to_string())
else:
    print('\n1. Computing analytical FoS for physics regularization...')
    new_fos = []
    for _, row in tqdm(units.iterrows(), total=len(units), desc='Analytical FoS'):
        soil = fetch_soil(row.center_lat, row.center_lon)
        fos = compute_fos_real(row.slope_degrees, soil['c'], soil['phi'], row['depth_real'], row['rain72h_climatic'])
        new_fos.append(fos)
    units['fos'] = new_fos

    print('\n2. Spatially mapping empirical historical landslide events to slope units...')
    # Convert lat/lon degrees to approximate local km coordinates for KDTree
    LAT_KM = 111.0
    lon_km = LAT_KM * np.cos(np.radians(units['center_lat'].mean()))

    unit_coords_km = np.column_stack([units['center_lat'] * LAT_KM, units['center_lon'] * lon_km])
    ls_coords_km   = np.column_stack([ls_points[:, 0] * LAT_KM, ls_points[:, 1] * lon_km])

    tree = cKDTree(ls_coords_km)
    distances_km, _ = tree.query(unit_coords_km, k=1)
    units['min_dist_to_landslide_km'] = np.round(distances_km, 2)

    # ── Ground Truth Label Assignment ──
    # RED (Failure = 1): Directly affected slope units (within 2.0 km of verified historical landslide)
    # GREEN (Stable = 0): Confirmed stable slope units (far from landslides and gentle/moderate slope)
    # ORANGE (Intermediate): Moderately steep slopes without recorded failures (held out from hard boundary)
    MATCH_RADIUS_KM = 2.0
    units['has_landslide_empirical'] = units['min_dist_to_landslide_km'] <= MATCH_RADIUS_KM

    new_risk = []
    for _, row in units.iterrows():
        if row['has_landslide_empirical']:
            new_risk.append('RED')
        elif row['slope_degrees'] < 18.0 or (row['slope_degrees'] < 28.0 and row['min_dist_to_landslide_km'] > 6.0):
            new_risk.append('GREEN')
        else:
            new_risk.append('ORANGE')

    units['risk_level'] = new_risk

    print('\nEmpirical Ground-Truth Risk Distribution:')
    print(units['risk_level'].value_counts().to_string())
    print('\nPer-state empirical landslide breakdown:')
    print(units.groupby(['region', 'risk_level']).size().unstack(fill_value=0).to_string())

    save_units_checkpoint()


In [ ]:
# ── STEP 4: Build the 9-feature dataset (Empirical Ground-Truth + Multi-Source Covariates) ─
FEAT_NAMES = ['Slope (°)', 'Cohesion (kPa)', 'Friction (°)', 'Depth (m)',
              'Saturation', 'PGA (g)', 'NDVI', 'Soil Type', 'Rainfall72h (mm)']

def build_features(df, label):
    X, regions = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'label={label}'):
        soil = fetch_soil(row.center_lat, row.center_lon)
        pga  = get_pga_cached(row.center_lat, row.center_lon)
        X.append([
            row.slope_degrees,
            soil['c'],
            soil['phi'],
            row['depth_real'],
            row['saturation_real'],
            pga,
            row['ndvi_real'],
            soil['soil'],
            row['rain72h_climatic'],
        ])
        regions.append(row['region'])
    y = np.full(len(df), label)
    return np.array(X), y, np.array(regions)

print(f'RED (Empirical Landslide Units):  {(units.risk_level=="RED").sum():,}')
print(f'GREEN (Confirmed Stable Units):  {(units.risk_level=="GREEN").sum():,}')
print(f'ORANGE (Intermediate / Held-out): {(units.risk_level=="ORANGE").sum():,}')

# Force rebuild to ensure empirical ground-truth is encoded
red_units   = units[units.risk_level == 'RED'].reset_index(drop=True)
green_units = units[units.risk_level == 'GREEN'].reset_index(drop=True)

print('\nBuilding failure-class features from real landslide units...')
X_pos, y_pos, reg_pos = build_features(red_units, 1)
print('\nBuilding stable-class features from confirmed stable terrain...')
X_neg, y_neg, reg_neg = build_features(green_units, 0)

X_all = np.vstack([X_pos, X_neg])
y_all = np.concatenate([y_pos, y_neg])
regions_all = np.concatenate([reg_pos, reg_neg])

np.random.seed(42)
shuffler = np.random.permutation(len(X_all))
X_all, y_all, regions_all = X_all[shuffler], y_all[shuffler], regions_all[shuffler]

np.savez(FEATURES_CKPT, X_all=X_all, y_all=y_all, regions_all=regions_all)
print(f'💾 Feature dataset checkpoint saved: {FEATURES_CKPT}')

print(f'\n📊 Dataset: {len(X_all):,} total units | {y_all.mean()*100:.1f}% positive failures')
print('Feature standard deviations (verifying non-zero variance):')
for name, s in zip(FEAT_NAMES, X_all.std(0)):
    print(f'  {name:<20}: std = {s:.3f}')


In [ ]:
# ── STEP 5: Exploratory Data Analysis ────────────────────────────────────────
df_eda = pd.DataFrame(X_all, columns=FEAT_NAMES)
df_eda['Label'] = y_all

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
fig.suptitle('Feature Distributions: Stable vs Failed Slopes', fontsize=16, fontweight='bold')
for ax, feat in zip(axes.flat, FEAT_NAMES):
    stable = df_eda[df_eda.Label==0][feat].dropna()
    failed = df_eda[df_eda.Label==1][feat].dropna()
    ax.hist(stable, bins=40, alpha=0.6, color='#2196F3', label='Stable', density=True)
    ax.hist(failed, bins=40, alpha=0.6, color='#F44336', label='Failed', density=True)
    ax.set_title(feat, fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature Statistics:')
print(df_eda.groupby('Label')[FEAT_NAMES].mean().T.rename(columns={0:'Stable Mean', 1:'Failed Mean'}).round(2))

In [ ]:
# ── STEP 6: Advanced PINN Architecture ───────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.block(x))

class AdvancedLandslidePINN(nn.Module):
    """
    9-input physics-constrained network with residual connections.
    Inputs: [slope, cohesion, friction, depth, saturation, PGA, NDVI, soil_type, rainfall_72h]
    Output: failure probability in [0,1]
    """
    def __init__(self, input_mean, input_std, dropout=0.2):
        super().__init__()
        # Safe normalization buffer: ensure sigma is bounded from below (min 0.05) to prevent numerical explosion
        safe_std = np.maximum(np.array(input_std, dtype=np.float32), 0.05)
        self.register_buffer('mu',    torch.tensor(input_mean, dtype=torch.float32))
        self.register_buffer('sigma', torch.tensor(safe_std,   dtype=torch.float32))

        self.stem = nn.Sequential(nn.Linear(9, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(dropout))
        self.res1 = ResBlock(128, dropout)
        self.res2 = ResBlock(128, dropout)
        self.head = nn.Sequential(
            nn.Linear(128,  64), nn.BatchNorm1d(64),  nn.GELU(), nn.Dropout(dropout/2),
            nn.Linear( 64,  32), nn.BatchNorm1d(32),  nn.GELU(),
            nn.Linear( 32,   1), nn.Sigmoid()
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x_n = (x - self.mu) / (self.sigma + 1e-6)
        return self.head(self.res2(self.res1(self.stem(x_n))))

    def predict_uncertainty(self, x, n=150):
        self.train()
        for module in self.modules():
            if isinstance(module, nn.BatchNorm1d):
                module.eval()
        with torch.no_grad():
            preds = torch.stack([self(x) for _ in range(n)])
        self.eval()
        return preds.mean(0).squeeze(), preds.std(0).squeeze()

def dual_physics_loss(xb, out, sharpness=6.0):
    """Combined Static Infinite-Slope + Seismic Newmark physics supervision."""
    slope, c, phi, z, pga, rain72h = xb[:,0], xb[:,1], xb[:,2], xb[:,3], xb[:,5], xb[:,8]
    GAMMA, GAMMA_W = 18.0, 9.81
    m = torch.clamp(rain72h / 150.0, max=1.0)
    b, p = torch.deg2rad(slope), torch.deg2rad(phi)
    num = c + (GAMMA - m*GAMMA_W)*z*torch.cos(b)**2*torch.tan(p)
    den = GAMMA*z*torch.sin(b)*torch.cos(b) + 1e-6
    fos_s  = num / den
    fos_eq = fos_s - pga * torch.tan(b)
    t_s  = torch.sigmoid(-sharpness*(fos_s  - 1.0))
    t_eq = torch.sigmoid(-sharpness*(fos_eq - 1.0))
    target = 0.6*t_s + 0.4*t_eq
    return nn.MSELoss()(out.squeeze(), target)

def compute_fos_np(X):
    slope, c, phi, z, pga, rain72h = X[:,0], X[:,1], X[:,2], X[:,3], X[:,5], X[:,8]
    m = np.clip(rain72h / 150.0, 0, 1.0)
    b, p = np.radians(slope), np.radians(phi)
    fos_s = (c + (18.0 - m*9.81)*z*np.cos(b)**2*np.tan(p)) / (18.0*z*np.sin(b)*np.cos(b) + 1e-6)
    return fos_s, fos_s - pga*np.tan(b)

n_params = sum(p.numel() for p in AdvancedLandslidePINN(np.zeros(9), np.ones(9)).parameters())
print(f'✅ Architecture: 9 → ResNet(128×2) → 64 → 32 → 1')
print(f'   Total parameters: {n_params:,}')


In [ ]:
# ── STEP 7: Data Split & DataLoaders (+ spatial holdout for true generalization) ─
# In addition to the usual random train/val/test split, this holds out TWO ENTIRE
# STATES that the model never sees during training at all — a proper spatial
# generalization test. Random row-level splits can look artificially good on
# spatial data because nearby slope units share very similar features (spatial
# autocorrelation); a state the model has genuinely never seen is a much harder,
# more honest test of whether it learned real physical relationships.
SPATIAL_HOLDOUT_STATES = ['sikkim', 'tripura']  # excluded from training entirely

spatial_mask = np.isin(regions_all, SPATIAL_HOLDOUT_STATES)
X_spatial_holdout = X_all[spatial_mask]
y_spatial_holdout = y_all[spatial_mask]

X_trainable = X_all[~spatial_mask]
y_trainable = y_all[~spatial_mask]

print(f'Spatial holdout ({", ".join(SPATIAL_HOLDOUT_STATES)}): {len(X_spatial_holdout):,} units — never seen during training')
print(f'Trainable pool (other 6 states): {len(X_trainable):,} units')

X_tr_np, X_tmp, y_tr_np, y_tmp = train_test_split(X_trainable, y_trainable, test_size=0.30, random_state=42, stratify=y_trainable)
X_va_np, X_te_np, y_va_np, y_te_np = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

input_mean = X_tr_np.mean(0)
input_std  = X_tr_np.std(0)

def to_t(X, y):
    return torch.tensor(X, dtype=torch.float32).to(device), torch.tensor(y, dtype=torch.float32).to(device)

X_tr,y_tr = to_t(X_tr_np, y_tr_np)
X_va,y_va = to_t(X_va_np, y_va_np)
X_te,y_te = to_t(X_te_np, y_te_np)
X_sh,y_sh = to_t(X_spatial_holdout, y_spatial_holdout)  # spatial holdout, for later

train_loader = DataLoader(TensorDataset(X_tr,y_tr), batch_size=256, shuffle=True,  drop_last=True)
val_loader   = DataLoader(TensorDataset(X_va,y_va), batch_size=512, shuffle=False)

# Class weights (inverse frequency, normalized to average 1.0) -- since the
# dataset is now the real, naturally-imbalanced class counts rather than
# synthetically balanced via oversampling, imbalance is handled here instead.
n_pos_tr = float((y_tr_np == 1).sum())
n_neg_tr = float((y_tr_np == 0).sum())
POS_WEIGHT = (n_pos_tr + n_neg_tr) / (2 * n_pos_tr)
NEG_WEIGHT = (n_pos_tr + n_neg_tr) / (2 * n_neg_tr)

print(f'\nTrain={len(X_tr_np):,} | Val={len(X_va_np):,} | Test (same 6 states)={len(X_te_np):,} | Spatial holdout={len(X_spatial_holdout):,}')
print(f'Train class balance: {n_pos_tr:.0f} failed / {n_neg_tr:.0f} stable ({100*n_pos_tr/(n_pos_tr+n_neg_tr):.1f}% positive)')
print(f'Class weights -> positive: {POS_WEIGHT:.3f}, negative: {NEG_WEIGHT:.3f}')
print(f'Input mean : {input_mean.round(2)}')
print(f'Input std  : {input_std.round(2)}')

In [ ]:
# ── STEP 8: Training with Adaptive Physics Warmup (500 epochs) ───────────────
EPOCHS          = 500
LR              = 1e-3
PATIENCE        = 50
LAM_MAX         = 0.5
WARMUP          = 100

model     = AdvancedLandslidePINN(input_mean, input_std, dropout=0.2).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

hist = {k:[] for k in ['train','val','data','phys','auc']}
best_val, patience_cnt, best_w = float('inf'), 0, None

print(f'Training on {device} for up to {EPOCHS} epochs')
print('='*70)
print(f'{"Epoch":>6} | {"Train":>8} | {"Val":>8} | {"DataL":>8} | {"PhysL":>8} | {"AUC":>6}')
print('-'*70)

for epoch in range(1, EPOCHS+1):
    lam = LAM_MAX * min(1.0, epoch/WARMUP)
    model.train()
    ep_d = ep_p = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        out  = model(xb)
        batch_w = torch.where(yb == 1, POS_WEIGHT, NEG_WEIGHT)
        ld   = nn.BCELoss(weight=batch_w)(out.squeeze(), yb)
        lp   = dual_physics_loss(xb, out)
        loss = ld + lam*lp
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_d += ld.item(); ep_p += lp.item()

    avg_d = ep_d/len(train_loader)
    avg_p = ep_p/len(train_loader)
    avg_t = avg_d + lam*avg_p

    model.eval()
    vp, vt, vl = [], [], 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            out = model(xb)
            batch_w = torch.where(yb == 1, POS_WEIGHT, NEG_WEIGHT)
            vl += (nn.BCELoss(weight=batch_w)(out.squeeze(), yb) + lam*dual_physics_loss(xb, out)).item()
            vp += out.squeeze().cpu().tolist()
            vt += yb.cpu().tolist()
    avg_v = vl/len(val_loader)
    auc   = roc_auc_score(vt, vp)
    scheduler.step()

    for k,v in zip(['train','val','data','phys','auc'],[avg_t,avg_v,avg_d,avg_p,auc]):
        hist[k].append(v)

    if avg_v < best_val:
        best_val=avg_v; best_w={k:v.clone() for k,v in model.state_dict().items()}; patience_cnt=0
    else:
        patience_cnt += 1

    if epoch%50==0 or epoch==1:
        print(f'{epoch:>6} | {avg_t:>8.4f} | {avg_v:>8.4f} | {avg_d:>8.4f} | {avg_p:>8.4f} | {auc:>6.4f}')
    if patience_cnt >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

model.load_state_dict(best_w)
print(f'\n✅ Training complete! Best val AUC: {max(hist["auc"]):.4f}')

torch.save({'model_state_dict': model.state_dict(),
            'input_mean': input_mean, 'input_std': input_std,
            'feat_names': FEAT_NAMES, 'best_val_auc': max(hist['auc']),
            'trained_from': 'lithos_all_ne_slope_units_final.gpkg'},
           'pinn_model_v2.pth')
print('✅ Checkpoint saved as pinn_model_v2.pth')

In [ ]:
# ── STEP 9: Training Curves ───────────────────────────────────────────────────
ep = range(1, len(hist['train'])+1)
fig, axes = plt.subplots(1,3,figsize=(16,4))
fig.suptitle('Phase 11 PINN Training History (NE-wide GPKG)', fontsize=14, fontweight='bold')

axes[0].plot(ep, hist['train'], label='Train', color='#2196F3')
axes[0].plot(ep, hist['val'],   label='Val',   color='#F44336')
axes[0].set_title('Total Loss'); axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_yscale('log')

axes[1].plot(ep, hist['data'], label='Data Loss',    color='#4CAF50')
axes[1].plot(ep, hist['phys'], label='Physics Loss', color='#FF9800')
axes[1].set_title('Data vs Physics Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, hist['auc'], color='#9C27B0')
axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[2].fill_between(ep, 0.5, hist['auc'], alpha=0.1, color='#9C27B0')
axes[2].set_title('Validation AUC'); axes[2].set_ylim([0.4,1.0]); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('training_curves.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ── STEP 10: Full Test Set Evaluation ────────────────────────────────────────
model.eval()
with torch.no_grad():
    probs = model(X_te).squeeze().cpu().numpy()
labels = y_te.cpu().numpy()
preds  = (probs >= 0.5).astype(int)
auc    = roc_auc_score(labels, probs)

print('='*50)
print(f'ROC-AUC : {auc:.4f}')
print(classification_report(labels, preds, target_names=['Stable','Failed']))

fig, axes = plt.subplots(2,2,figsize=(13,11))
fig.suptitle('Phase 11 Evaluation Dashboard (NE-wide)', fontsize=15, fontweight='bold')

fpr,tpr,_ = roc_curve(labels, probs)
axes[0,0].plot(fpr,tpr,color='#2196F3',lw=2,label=f'AUC={auc:.3f}')
axes[0,0].plot([0,1],[0,1],'k--',alpha=0.4); axes[0,0].fill_between(fpr,tpr,alpha=0.1,color='#2196F3')
axes[0,0].set_xlabel('FPR'); axes[0,0].set_ylabel('TPR'); axes[0,0].set_title('ROC Curve')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

prec,rec,_ = precision_recall_curve(labels,probs)
axes[0,1].plot(rec,prec,color='#4CAF50',lw=2)
axes[0,1].axhline(labels.mean(),color='gray',linestyle='--',label='Baseline')
axes[0,1].set_xlabel('Recall'); axes[0,1].set_ylabel('Precision'); axes[0,1].set_title('PR Curve')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

cm = confusion_matrix(labels,preds)
axes[1,0].imshow(cm,cmap='Blues')
axes[1,0].set_xticks([0,1]); axes[1,0].set_yticks([0,1])
axes[1,0].set_xticklabels(['Stable','Failed']); axes[1,0].set_yticklabels(['Stable','Failed'])
axes[1,0].set_xlabel('Predicted'); axes[1,0].set_ylabel('Actual')
for i in range(2):
    for j in range(2):
        axes[1,0].text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=16,fontweight='bold',
                       color='white' if cm[i,j]>cm.max()/2 else 'black')
axes[1,0].set_title('Confusion Matrix')

fp,mp = calibration_curve(labels,probs,n_bins=10)
axes[1,1].plot(mp,fp,'s-',color='#F44336',lw=2,label='PINN v2 (NE)')
axes[1,1].plot([0,1],[0,1],'k--',alpha=0.5,label='Perfect')
axes[1,1].set_xlabel('Mean Predicted Prob'); axes[1,1].set_ylabel('Fraction Positives')
axes[1,1].set_title('Calibration Curve'); axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('evaluation.png', dpi=150, bbox_inches='tight'); plt.show()

## 🗺️ Spatial holdout evaluation — the real generalization test
The test AUC above comes from the SAME 6 states the model trained on (random
row-level split), so it can look better than the model's true skill on genuinely
unseen terrain. This evaluates on Sikkim and Tripura, which the model never saw
at all during training — a much more honest measure of real-world performance.


In [ ]:
model.eval()
with torch.no_grad():
    probs_sh = model(X_sh).squeeze().cpu().numpy()
labels_sh = y_sh.cpu().numpy()
preds_sh  = (probs_sh >= 0.5).astype(int)

if len(np.unique(labels_sh)) > 1:
    auc_sh = roc_auc_score(labels_sh, probs_sh)
    print('='*60)
    print(f'SPATIAL HOLDOUT AUC ({", ".join(SPATIAL_HOLDOUT_STATES)}): {auc_sh:.4f}')
    print('='*60)
    print(classification_report(labels_sh, preds_sh, target_names=['Stable','Failed']))
    print('\nCompare this to the in-distribution test AUC above. A meaningfully lower')
    print('spatial-holdout AUC is normal and expected — it tells you how much the model')
    print('relies on region-specific patterns vs. genuinely transferable physical')
    print('relationships. A gap of 0.05-0.15 is typical and healthy; a much larger gap')
    print('would suggest the model is still leaning on shortcuts rather than physics.')
else:
    print('Spatial holdout set has only one class present — cannot compute AUC.')
    print('(This can happen if one of the holdout states had very few RED or GREEN units.)')

In [ ]:
# ── STEP 11: Physics Consistency Check ───────────────────────────────────────
fos_s, fos_eq = compute_fos_np(X_te_np)

fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('Physics Consistency: FoS vs Predicted Probability', fontsize=13, fontweight='bold')

sc = axes[0].scatter(np.clip(fos_s,0,3), probs, c=labels, cmap='RdBu_r', s=6, alpha=0.45)
axes[0].axvline(1.0,color='red',linestyle='--',label='FoS=1.0')
axes[0].axvline(1.5,color='orange',linestyle='--',label='FoS=1.5')
axes[0].axhline(0.5,color='black',linestyle=':',alpha=0.5)
plt.colorbar(sc,ax=axes[0],label='True Label')
axes[0].set_xlabel('Static Factor of Safety'); axes[0].set_ylabel('Predicted Failure Probability')
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_title('Static FoS vs Prob')

bins = np.linspace(0.2,3.0,20)
bidx = np.digitize(fos_s, bins)
bm   = [probs[bidx==i].mean() if (bidx==i).sum()>0 else np.nan for i in range(len(bins))]
axes[1].plot(bins,bm,'o-',color='#9C27B0',lw=2)
axes[1].axvline(1.0,color='red',linestyle='--',label='FoS=1.0')
axes[1].axvline(1.5,color='orange',linestyle='--',label='FoS=1.5')
axes[1].set_xlabel('Static Factor of Safety'); axes[1].set_ylabel('Mean Failure Prob')
axes[1].set_ylim([0,1]); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title('Average Probability by FoS Band')

plt.tight_layout(); plt.savefig('physics_check.png', dpi=150, bbox_inches='tight'); plt.show()

print(f'Physics consistency:')
print(f'  FoS < 1.0  → mean prob = {probs[fos_s<1.0].mean():.3f}  (should be >0.75)')
print(f'  1.0 < FoS < 1.5 → mean prob = {probs[(fos_s>=1.0)&(fos_s<1.5)].mean():.3f}  (should be ~0.4-0.6)')
print(f'  FoS > 1.5  → mean prob = {probs[fos_s>1.5].mean():.3f}  (should be <0.25)')

In [ ]:
# ── STEP 12: MC Dropout Uncertainty ──────────────────────────────────────────
print('Running 150 MC Dropout inference passes...')
mu_p, sd_p = model.predict_uncertainty(X_te, n=150)
mu_p = mu_p.cpu().numpy(); sd_p = sd_p.cpu().numpy()
print(f'Mean uncertainty: {sd_p.mean():.4f}  |  Max: {sd_p.max():.4f}')

fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('MC Dropout Uncertainty Quantification', fontsize=13, fontweight='bold')

idx500 = np.argsort(mu_p)[:500]
axes[0].scatter(range(len(idx500)), mu_p[idx500], c=labels[idx500], cmap='RdBu_r', s=10, alpha=0.7)
axes[0].fill_between(range(len(idx500)),
                     mu_p[idx500]-2*sd_p[idx500],
                     mu_p[idx500]+2*sd_p[idx500], alpha=0.2, color='orange', label='±2σ')
axes[0].axhline(0.5,color='k',linestyle='--',alpha=0.4)
axes[0].set_xlabel('Samples (sorted)'); axes[0].set_ylabel('Failure Probability')
axes[0].set_title('Predictions with Confidence Bands'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].hist(sd_p[labels==0],bins=40,alpha=0.6,color='#2196F3',density=True,label='Stable')
axes[1].hist(sd_p[labels==1],bins=40,alpha=0.6,color='#F44336',density=True,label='Failed')
axes[1].set_xlabel('Uncertainty (σ)'); axes[1].set_ylabel('Density')
axes[1].set_title('Uncertainty Distribution by Class'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('uncertainty.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ── STEP 13: Sensitivity Analysis ────────────────────────────────────────────
base = [32, 12, 27, 3.5, 0.65, 0.28, 0.45, 0.50, 140]
ranges = [np.linspace(5,65,60), np.linspace(1,45,60), np.linspace(10,45,60),
          np.linspace(0.5,10,60), np.linspace(0,1,60),  np.linspace(0,0.6,60),
          np.linspace(0,1,60),    np.linspace(0,1,60),   np.linspace(0,500,60)]

fig, axes = plt.subplots(1,9,figsize=(26,4))
fig.suptitle('Sensitivity Analysis — Effect of Each Feature on Failure Probability', fontsize=13, fontweight='bold')
model.eval()
for i,(feat,rng,ax) in enumerate(zip(FEAT_NAMES,ranges,axes)):
    probs_s = []
    for v in rng:
        s = base.copy(); s[i] = v
        with torch.no_grad():
            probs_s.append(model(torch.tensor([s],dtype=torch.float32).to(device)).item())
    ax.plot(rng, probs_s, lw=2, color='#9C27B0')
    ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, lw=1)
    ax.axvline(base[i], color='blue', linestyle=':', alpha=0.7)
    ax.fill_between(rng, 0, probs_s, alpha=0.1, color='#9C27B0')
    ax.set_xlabel(feat, fontsize=7); ax.set_ylim([0,1]); ax.grid(alpha=0.3)
    if i==0: ax.set_ylabel('Failure Prob')

plt.tight_layout(); plt.savefig('sensitivity.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ── STEP 14: Real Scenario Inference ─────────────────────────────────────────
print('='*90)
print(f'{"Scenario":<22}|{"Slp":>5}|{"C":>5}|{"phi":>5}|{"z":>5}|{"Sat":>5}|{"PGA":>5}|{"FoS_s":>6}|{"Prob":>6}|{"±σ":>5}| Risk')
print('-'*90)

# Realistic extreme vs baseline scenarios aligned to CHIRPS rainfall & ERA5 saturation scales:
scenarios = [
    # (name,               slope, c,  phi,  z,   sat,  pga,  ndvi, soil, rf72)
    ('Wayanad Extreme',     42,   8,  21,  4.5, 0.95, 0.20, 0.35, 0.58, 260),
    ('Sikkim Seismo-Rain',  38,   9,  24,  4.0, 0.85, 0.38, 0.40, 0.50, 180),
    ('Cherrapunji Monsoon', 35,  10,  25,  3.5, 0.92, 0.28, 0.35, 0.52, 280),
    ('Post-rain NH-29',     36,   9,  22,  4.0, 0.88, 0.32, 0.30, 0.55, 210),
    ('Nagaland Degraded',   32,  11,  24,  3.5, 0.80, 0.30, 0.25, 0.60, 175),
    ('Borderline Assam',    26,  14,  28,  3.0, 0.55, 0.25, 0.55, 0.45, 120),
    ('Rocky Arunachal',     24,  32,  38,  2.0, 0.35, 0.36, 0.68, 0.25,  80),
    ('Stable Munnar Hills', 16,  28,  36,  1.8, 0.30, 0.16, 0.80, 0.30,  50),
]

for sc in scenarios:
    name = sc[0]; feats = list(sc[1:])
    xt = torch.tensor([feats], dtype=torch.float32).to(device)

    # Uncertainty estimation
    mu_i, sd_i = model.predict_uncertainty(xt, n=200)
    mu_i = mu_i.item() if mu_i.ndim == 0 else mu_i[0].item()
    sd_i = sd_i.item() if sd_i.ndim == 0 else sd_i[0].item()

    fos_val, _ = compute_fos_np(np.array([feats]))
    fos_val = fos_val[0]

    risk = 'HIGH  🔴' if mu_i>0.65 else ('MED 🟠' if mu_i>0.35 else 'LOW  🟢')
    print(f'{name:<22}|{feats[0]:>5.0f}|{feats[1]:>5.0f}|{feats[2]:>5.0f}|{feats[3]:>5.1f}|{feats[4]:>5.2f}|{feats[5]:>5.2f}|{fos_val:>6.2f}|{mu_i:>6.3f}|{sd_i:>5.3f}| {risk}')


## 🏗️ IS 14680:1999 — Landslide Classification + Recommended Control Measure
Extends risk *prediction* into an actionable engineering *recommendation*, per Table 1
of the Bureau of Indian Standards' "Landslide Control — Guidelines". Given a slope
unit's physical characteristics, this classifies the likely movement type and material
class, then looks up BIS's recommended control measure for that combination.

**Important caveat, stated directly in the standard itself (Section 5):** IS 14680
requires actual subsoil investigation, cone penetration testing, and engineering
property testing (per IS 1498 and IS 1892) before selecting a real remedial design.
The classification below is a **first-pass screening heuristic** built from available
remote-sensing-derived features (slope, estimated soil depth, texture, saturation) —
it prioritizes which slopes warrant field investigation and roughly which measure
category to budget for. It is not a substitute for the field investigation the
standard itself mandates.


In [ ]:
# ── IS 14680:1999 Table 1 — Movement Type × Material Type → Control Measure ──
CONTROL_MEASURE_TABLE = {
    ('falls',        'fine'):   ('Earth fall',         'Geotextile nailed on slope/spot bolting'),
    ('falls',        'coarse'): ('Debris fall',         'Geotextile nailed on slope/spot bolting'),
    ('falls',        'rock'):   ('Rock fall',           'Geotextile nailed on slope/spot bolting'),
    ('topples',      'fine'):   ('Earth topple',        'Breast walls/soil nailing'),
    ('topples',      'coarse'): ('Debris topple',       'Breast walls/soil nailing'),
    ('topples',      'rock'):   ('Rock topple',         'Breast walls/soil nailing'),
    ('rotational',   'fine'):   ('Earth slump',         'Alteration of slope profile and earth/rock fill buttress'),
    ('rotational',   'coarse'): ('Debris slump',        'Alteration of slope profile and earth/rock fill buttress'),
    ('rotational',   'rock'):   ('Rock slump',          'Alteration of slope profile and earth/rock fill buttress'),
    ('translational','fine'):   ('Earth block slide',   'Reinforced earth or rock reinforcement in rock slope'),
    ('translational','coarse'): ('Debris block slide',  'Reinforced earth or rock reinforcement in rock slope'),
    ('translational','rock'):   ('Rock block slide',    'Reinforced earth or rock reinforcement in rock slope'),
    ('slide',        'fine'):   ('Earth slide',         'Biotechnical measures'),
    ('slide',        'coarse'): ('Debris slide',        'Biotechnical measures'),
    ('slide',        'rock'):   ('Rock slide',          'Biotechnical measures'),
    ('lateral_spread','fine'):  ('Earth spread',        'Check dams along gully'),
    ('lateral_spread','coarse'):('Debris spread',       'Check dams along gully'),
    ('lateral_spread','rock'):  ('Rock spread',         'Check dams along gully'),
    ('flow',         'fine'):   ('Earth flow',          'Series of check dams'),
    ('flow',         'coarse'): ('Debris flow',         'Series of check dams'),
    ('flow',         'rock'):   ('Rock flow',           'Series of check dams'),
    ('creep',        'fine'):   ('Soil creep',          'Rows of deep piles'),
    ('creep',        'coarse'): ('Deep creep',          'Rows of deep piles'),
    ('complex',      'any'):    ('Complex (combined movement types)', 'Combined system'),
}

def classify_material(soil_type_frac, depth_m, phi_deg):
    """Approximates IS 14680's fine soil / coarse soil / bedrock material classes
    from available covariates. soil_type_frac is the SoilGrids clay fraction (0-1)."""
    if depth_m < 1.0 or phi_deg > 38:
        return 'rock'
    return 'fine' if soil_type_frac >= 0.35 else 'coarse'

def classify_movement(slope_deg, sat, depth_m, mc_uncertainty=None):
    """Heuristic movement-type classification from slope angle, saturation, and
    failure depth. This is a first-pass screening approximation, not a substitute
    for field geological mapping (which IS 14680 itself requires — see Section 5)."""
    # High model uncertainty -> flag as complex/indeterminate rather than force a guess
    if mc_uncertainty is not None and mc_uncertainty > 0.25:
        return 'complex'
    if sat > 0.80:
        return 'flow'
    if slope_deg > 50:
        return 'falls'
    if slope_deg > 42:
        return 'topples'
    if slope_deg > 30 and depth_m > 4.0 and sat > 0.55:
        return 'rotational'
    if 22 <= slope_deg <= 38 and depth_m <= 4.0:
        return 'translational'
    if slope_deg < 20 and sat > 0.5:
        return 'lateral_spread'
    if depth_m < 1.5 and slope_deg < 25:
        return 'creep'
    return 'slide'

def recommend_control_measure(slope_deg, soil_type_frac, phi_deg, depth_m, sat, mc_uncertainty=None):
    movement = classify_movement(slope_deg, sat, depth_m, mc_uncertainty)
    material = classify_material(soil_type_frac, depth_m, phi_deg) if movement != 'complex' else 'any'
    key = (movement, material) if (movement, material) in CONTROL_MEASURE_TABLE else ('complex', 'any')
    is_category, measure = CONTROL_MEASURE_TABLE[key]
    return {
        'movement_type': movement,
        'material_class': material,
        'is14680_category': is_category,
        'recommended_measure': measure,
    }

print('✅ IS 14680:1999 classification + control measure lookup ready')
print(f'   {len(CONTROL_MEASURE_TABLE)} movement×material combinations mapped from Table 1')

### Apply IS 14680 recommendations to the scenario list above

In [ ]:
print(f'{"Scenario":<22}|{"Movement":<15}|{"Material":<8}|{"IS14680 Category":<20}| Recommended Measure')
print('-'*115)
for sc in scenarios:
    name = sc[0]; feats = list(sc[1:])
    slope_deg, c_kpa, phi_deg, depth_m, sat, pga, ndvi, soil_type_frac, rf72 = feats
    rec = recommend_control_measure(slope_deg, soil_type_frac, phi_deg, depth_m, sat)
    print(f'{name:<22}|{rec["movement_type"]:<15}|{rec["material_class"]:<8}|{rec["is14680_category"]:<20}| {rec["recommended_measure"]}')

print('\nNote: this uses the manually-specified scenario features directly (not model')
print('uncertainty), since these are illustrative examples rather than real inference')
print('outputs. For real slope units, pass predict_uncertainty()\'s sigma as')
print('mc_uncertainty to flag high-disagreement cases as \'complex\' rather than force')
print('a potentially wrong single-category guess.')

In [ ]:
# ── STEP 15: Download Trained Model ──────────────────────────────────────────
from google.colab import files

print('📊 Final Summary')
print('='*50)
print(f'Architecture  : 9 → ResNet(128×2) → 64 → 32 → 1')
print(f'Physics       : Static Infinite Slope + Seismic Newmark (60/40 weighted)')
print(f'Trained from  : lithos_all_ne_slope_units_final.gpkg ({len(units):,} slope units)')
print(f'Best Val AUC  : {max(hist["auc"]):.4f}')
print(f'Test AUC      : {roc_auc_score(labels, probs):.4f}')
print(f'Total Params  : {sum(p.numel() for p in model.parameters()):,}')
print(f'Checkpoint    : pinn_model_v2.pth')
print('='*50)

# Also copy straight to Drive so it's saved even without a manual download
DRIVE_MODEL_DIR = '/content/drive/MyDrive/LITHOS/Phase11_data'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
import shutil
shutil.copy('pinn_model_v2.pth', f'{DRIVE_MODEL_DIR}/pinn_model_v2.pth')
print(f'✅ Also saved to Drive: {DRIVE_MODEL_DIR}/pinn_model_v2.pth')

print('\nDownloading pinn_model_v2.pth to your computer...')
print('👉 After downloading, put it in: LITHOS/phase7-webapp/backend/')
files.download('pinn_model_v2.pth')

## 🏔️ 3D Terrain Mesh + Risk Heatmap (continuous surface, not flat polygons)
Renders a real DEM as a 3D surface for a zoomed-in region you choose, with your
PINN's predicted failure probability draped over it as a continuous color heatmap
— interpolated from your slope units onto the same grid the terrain uses, so the
hazard coloring actually follows the real terrain shape underneath it.

**Pick a region below** — small enough to render in full detail (roughly 0.3-0.5°
square, ~30-50km across). Whole-NE-India at 30m resolution would be too much
geometry for one interactive plot; a zoomed-in view is also more useful for a
report or demo, matching the reference image's single-valley scope.


In [ ]:
!pip install plotly -q
import plotly.graph_objects as go
from scipy.interpolate import griddata

# ── Pick a region to render (edit these bounds) ──────────────────────────────
# Default: a slice of the Teesta valley area in Sikkim, a real documented
# landslide/GLOF-prone corridor -- change to zoom in on any area you want.
VIZ_SOUTH, VIZ_WEST, VIZ_NORTH, VIZ_EAST = 27.15, 88.45, 27.45, 88.75
VIZ_GRID_SIZE = 400  # target pixels per side; actually used now (see below) -- try 200-500. Values needing <30m/pixel are capped at native DEM resolution.

viz_region = ee.Geometry.Rectangle([VIZ_WEST, VIZ_SOUTH, VIZ_EAST, VIZ_NORTH])

# ── Actually wire up VIZ_GRID_SIZE (previously defined but unused -- a real bug:
# sampleRectangle() was just pulling native 30m pixels with zero resolution
# control, and no protection against exceeding Earth Engine's array-transfer
# limits for larger regions). This reprojects to a scale computed from the
# requested grid size, and caps it at the DEM's native 30m -- you cannot get
# genuinely finer detail than the source data actually contains; requesting a
# scale below 30m would just interpolate fake pixels, not add real resolution.
region_width_m = (VIZ_EAST - VIZ_WEST) * 111000 * np.cos(np.radians((VIZ_NORTH + VIZ_SOUTH) / 2))
target_scale_m = max(30, region_width_m / VIZ_GRID_SIZE)
expected_pixels_per_side = int(region_width_m / target_scale_m)

print(f'Region width: ~{region_width_m/1000:.1f} km | Target scale: {target_scale_m:.0f} m/pixel '
      f'(native DEM floor: 30 m) | Expected grid: ~{expected_pixels_per_side}x{expected_pixels_per_side}')
if expected_pixels_per_side > 600:
    print(f'⚠️  Grid may be large enough to hit Earth Engine transfer limits. If this cell fails or')
    print(f'   hangs, lower VIZ_GRID_SIZE or shrink the bounding box.')

print('Fetching real DEM for the selected region...')
dem_viz = (ee.ImageCollection('COPERNICUS/DEM/GLO30_2024_1')
             .filterBounds(viz_region)
             .select('DEM')
             .mosaic()
             .clip(viz_region)
             .reproject(crs='EPSG:4326', scale=target_scale_m))

dem_sample = dem_viz.sampleRectangle(region=viz_region, defaultValue=0)
dem_array = np.array(dem_sample.get('DEM').getInfo(), dtype=float)
print(f'DEM grid shape: {dem_array.shape} (~{target_scale_m:.0f} m/pixel)')

# Build lat/lon coordinate grids matching the DEM array's actual shape
rows, cols = dem_array.shape
lat_grid = np.linspace(VIZ_NORTH, VIZ_SOUTH, rows)
lon_grid = np.linspace(VIZ_WEST, VIZ_EAST, cols)
lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)

# ── Predict failure probability for every unit that falls in this region ─────
region_units = units[
    (units.center_lat >= VIZ_SOUTH) & (units.center_lat <= VIZ_NORTH) &
    (units.center_lon >= VIZ_WEST)  & (units.center_lon <= VIZ_EAST)
].copy()
print(f'{len(region_units)} slope units found in the selected region')

if len(region_units) < 4:
    print('⚠️ Too few units in this region to interpolate a heatmap -- pick a larger')
    print('   or different bounding box (try a state you know has good unit density).')
else:
    model.eval()
    feats_list = []
    for _, row in region_units.iterrows():
        soil = fetch_soil(row.center_lat, row.center_lon)
        pga  = get_pga_cached(row.center_lat, row.center_lon)
        feats_list.append([
            row.slope_degrees, soil['c'], soil['phi'], row['depth_real'],
            row['saturation_real'], pga, row['ndvi_real'], soil['soil'], row['rain72h_climatic'],
        ])
    xt = torch.tensor(feats_list, dtype=torch.float32).to(device)
    with torch.no_grad():
        region_probs = model(xt).squeeze().cpu().numpy()
    if region_probs.ndim == 0:
        region_probs = np.array([region_probs.item()])

    # Interpolate the point-wise probabilities onto the DEM's full grid
    points = np.column_stack([region_units.center_lon.to_numpy(), region_units.center_lat.to_numpy()])
    risk_grid = griddata(points, region_probs, (lon_mesh, lat_mesh), method='linear')
    # Fill any extrapolation gaps (outside the convex hull of your points) with nearest-neighbor
    nan_mask = np.isnan(risk_grid)
    if nan_mask.any():
        risk_grid[nan_mask] = griddata(points, region_probs, (lon_mesh[nan_mask], lat_mesh[nan_mask]), method='nearest')

    # ── Render as an interactive 3D surface ───────────────────────────────────
    fig = go.Figure(data=[go.Surface(
        z=dem_array,
        surfacecolor=risk_grid,
        colorscale=[[0.0, '#2A9D8F'], [0.5, '#F4A261'], [1.0, '#E63946']],  # green -> orange -> red
        cmin=0, cmax=1,
        colorbar=dict(title='Failure<br>Probability', tickformat='.0%'),
        lighting=dict(ambient=0.6, diffuse=0.8, specular=0.2),
        contours_z=dict(show=False),
    )])

    fig.update_layout(
        title=f'LITHOS — 3D Terrain + Failure Probability Heatmap<br><sub>({VIZ_SOUTH:.2f}-{VIZ_NORTH:.2f}°N, {VIZ_WEST:.2f}-{VIZ_EAST:.2f}°E)</sub>',
        scene=dict(
            xaxis_title='Longitude', yaxis_title='Latitude', zaxis_title='Elevation (m)',
            aspectratio=dict(x=1, y=1, z=0.3),
            camera=dict(eye=dict(x=1.4, y=-1.4, z=0.9)),
        ),
        width=950, height=700,
        margin=dict(l=0, r=0, t=80, b=0),
        paper_bgcolor='#0d1117',
        font=dict(color='white'),
    )
    fig.show()

    # Save a static image copy too, for reports/slides
    try:
        fig.write_image('lithos_3d_terrain_heatmap.png', scale=2)
        from google.colab import files
        files.download('lithos_3d_terrain_heatmap.png')
        print('✅ Saved and downloaded lithos_3d_terrain_heatmap.png')
    except Exception as e:
        print(f'(Static image export needs kaleido: !pip install -U kaleido -- interactive plot above still works. {e})')

## 📍 Priority Site Export — for commissioning targeted LiDAR/drone survey
Ranks every real slope unit by **risk × model uncertainty** — high probability of
failure AND high disagreement across the MC-Dropout ensemble means the model
itself isn't confident, which is exactly where real field investigation (per IS
14680 Section 5 / IS 1892) adds the most value. This is the handoff artifact:
instead of surveying all 255,000 km² at high resolution (impossible and
unnecessary), it gives survey teams a ranked, coordinate-precise shortlist.

**Note on resolution:** this uses the same 30m real terrain data as the rest of
the model — it identifies *where* to commission higher-resolution data collection,
it doesn't fabricate that resolution itself (see the earlier discussion on why
interpolated "fake detail" isn't used here).


In [ ]:
# ── Run inference + uncertainty across every real unit ────────────────────────
N_MC_SAMPLES = 100  # fewer than the 150-200 used elsewhere -- this runs across
                     # the full dataset, not just a handful of scenarios, so kept
                     # leaner for speed. Increase if you want tighter uncertainty
                     # estimates and have time to spare.

model.eval()
all_probs, all_sigmas = [], []
BATCH = 512

print(f'Running inference + MC-Dropout uncertainty on {len(units):,} real units...')
for start in tqdm(range(0, len(units), BATCH)):
    batch_rows = units.iloc[start:start+BATCH]
    feats_batch = []
    for _, row in batch_rows.iterrows():
        soil = fetch_soil(row.center_lat, row.center_lon)
        pga  = get_pga_cached(row.center_lat, row.center_lon)
        feats_batch.append([
            row.slope_degrees, soil['c'], soil['phi'], row['depth_real'],
            row['saturation_real'], pga, row['ndvi_real'], soil['soil'], row['rain72h_climatic'],
        ])
    xt = torch.tensor(feats_batch, dtype=torch.float32).to(device)
    mu, sd = model.predict_uncertainty(xt, n=N_MC_SAMPLES)
    mu = mu.cpu().numpy(); sd = sd.cpu().numpy()
    if mu.ndim == 0:
        mu, sd = np.array([mu.item()]), np.array([sd.item()])
    all_probs.extend(mu.tolist())
    all_sigmas.extend(sd.tolist())

units['pred_probability'] = all_probs
units['pred_uncertainty'] = all_sigmas
units['priority_score']   = units['pred_probability'] * units['pred_uncertainty']

print(f'\n✅ Inference complete: mean probability={units.pred_probability.mean():.3f}, '
      f'mean uncertainty={units.pred_uncertainty.mean():.3f}')

# ── Rank and export top priority sites ─────────────────────────────────────────
N_PRIORITY_SITES = 100

priority_sites = units.nlargest(N_PRIORITY_SITES, 'priority_score').copy()

# Attach IS 14680 recommended control measure to each priority site
recs = []
for _, row in priority_sites.iterrows():
    soil = fetch_soil(row.center_lat, row.center_lon)
    rec = recommend_control_measure(
        row.slope_degrees, soil['soil'], soil['phi'], row['depth_real'],
        row['saturation_real'], mc_uncertainty=row['pred_uncertainty']
    )
    recs.append(rec)

priority_export = pd.DataFrame({
    'rank':                 range(1, len(priority_sites) + 1),
    'unit_id':              priority_sites['unit_id'].values,
    'region':               priority_sites['region'].values,
    'latitude':             priority_sites['center_lat'].values,
    'longitude':            priority_sites['center_lon'].values,
    'slope_degrees':        priority_sites['slope_degrees'].round(1).values,
    'pred_failure_prob':    priority_sites['pred_probability'].round(3).values,
    'model_uncertainty':    priority_sites['pred_uncertainty'].round(3).values,
    'priority_score':       priority_sites['priority_score'].round(3).values,
    'current_risk_level':   priority_sites['risk_level'].values,
    'is14680_movement_type':  [r['is14680_category'] for r in recs],
    'is14680_recommended_measure': [r['recommended_measure'] for r in recs],
})

EXPORT_PATH = '/content/drive/MyDrive/LITHOS/Phase11_data/priority_survey_sites.csv'
priority_export.to_csv(EXPORT_PATH, index=False)

print(f'\n✅ Exported top {N_PRIORITY_SITES} priority sites: {EXPORT_PATH}')
print(f'\nPer-state breakdown of priority sites:')
print(priority_sites['region'].value_counts().to_string())
print(f'\nTop 10 highest-priority sites:')
print(priority_export.head(10).to_string(index=False))

from google.colab import files
files.download(EXPORT_PATH)